# 🐢🐇 Day 02: Linked List Two-Pointer — The Fast & Slow Dance

---

## The Most Versatile Linked List Trick

You've already seen Floyd's cycle detection. But fast/slow pointers solve way more than just cycle problems. Here's the beautiful thing:

- **Find the middle?** When fast reaches the end, slow is at the middle.
- **Find the nth node from end?** Start fast n steps ahead, then move both — when fast hits the end, slow is at the nth from end.
- **Find where a cycle starts?** After detection, reset one pointer to head and walk both at the same speed — they meet at the cycle start. (This is a math proof that blows everyone's mind.)

All of these are O(n) time, O(1) space. No extra data structures needed.

---

In [ ]:
# Reuse our helpers from Day 01
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build_list(values):
    dummy = ListNode(0)
    curr = dummy
    for v in values:
        curr.next = ListNode(v)
        curr = curr.next
    return dummy.next

def print_list(head):
    parts = []
    while head:
        parts.append(str(head.val))
        head = head.next
    print(" → ".join(parts) + " → None")

---

## 🎯 Problem 1: Middle of the Linked List (LC #876)

```
Input:  1 → 2 → 3 → 4 → 5  →  Output: node with value 3
Input:  1 → 2 → 3 → 4       →  Output: node with value 3 (second middle)
```

**The trick is beautiful:** Fast pointer moves 2x speed. When fast reaches the end, slow has traveled exactly half the distance — it's at the middle.

```
Step:  slow   fast
  0:   [1]    [1]
  1:   [2]    [3]
  2:   [3]    [5]  ← fast at end, slow at middle!
```

In [ ]:
def middleNode(head):
    """
    Find middle node using fast/slow pointers.
    Time: O(n), Space: O(1)
    """
    slow = fast = head
    
    while fast and fast.next:
        slow = slow.next       # 1 step
        fast = fast.next.next  # 2 steps
    
    return slow  # Slow is at the middle

# Odd length: 1→2→3→4→5
lst = build_list([1, 2, 3, 4, 5])
print(f"Middle of [1,2,3,4,5]: {middleNode(lst).val}")  # 3

# Even length: 1→2→3→4 → returns second middle
lst2 = build_list([1, 2, 3, 4])
print(f"Middle of [1,2,3,4]: {middleNode(lst2).val}")    # 3

---

## 🎯 Problem 2: Remove Nth Node From End of List (LC #19)

```
Input:  1 → 2 → 3 → 4 → 5, n = 2
Output: 1 → 2 → 3 → 5  (removed 4, which is 2nd from end)
```

**The idea:** If you send fast ahead by n steps first, then move both at the same speed, when fast reaches the end, slow will be exactly n nodes behind — right at the node to remove.

```
n = 2, list: 1 → 2 → 3 → 4 → 5

Start fast 2 ahead:   slow=[1]  fast=[3]
Move both:            slow=[2]  fast=[4]
Move both:            slow=[3]  fast=[5]
Move both:            slow=[3]  fast=None ← stop!
                      ↑ slow is just BEFORE the node to remove
```

We use a dummy node so slow can land right before the target, even if the target is the head.

In [ ]:
def removeNthFromEnd(head, n):
    """
    Remove nth node from end using two pointers.
    Time: O(n), Space: O(1)
    """
    dummy = ListNode(0)
    dummy.next = head
    slow = fast = dummy
    
    # Move fast n+1 steps ahead (so slow lands BEFORE target)
    for _ in range(n + 1):
        fast = fast.next
    
    # Move both until fast reaches end
    while fast:
        slow = slow.next
        fast = fast.next
    
    # slow.next is the node to remove
    slow.next = slow.next.next
    
    return dummy.next

lst = build_list([1, 2, 3, 4, 5])
result = removeNthFromEnd(lst, 2)
print("Remove 2nd from end:", end=" "); print_list(result)  # 1→2→3→5

# Edge case: remove head (n = length)
lst2 = build_list([1, 2])
result2 = removeNthFromEnd(lst2, 2)
print("Remove head:", end=" "); print_list(result2)  # 2

---

## 🎯 Problem 3: Linked List Cycle II — Find Cycle Start (LC #142)

Not just "is there a cycle?" but "WHERE does the cycle start?"

```
Input:  3 → 2 → 0 → -4 → back to 2
Output: Node with value 2 (the cycle start)
```

**The beautiful math:** After slow and fast meet inside the cycle, reset one of them to head. Now walk both at speed 1. They'll meet at the cycle start.

**Why does this work?** Let's say:
- Distance from head to cycle start = `a`
- Distance from cycle start to meeting point = `b`
- Remaining cycle length = `c`

When they meet: slow traveled `a + b`, fast traveled `a + b + b + c`.
Since fast = 2 × slow: `a + b + b + c = 2(a + b)` → `c = a`.

So the distance from the meeting point around the rest of the cycle equals the distance from head to cycle start. Walk both at speed 1 and they converge at the start!

In [ ]:
def detectCycle(head):
    """
    Find the node where the cycle begins.
    Time: O(n), Space: O(1)
    """
    slow = fast = head
    
    # Phase 1: Detect cycle (find meeting point)
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow == fast:
            break
    else:
        return None  # No cycle
    
    # Phase 2: Find cycle start
    # Reset one pointer to head, move both at speed 1
    slow = head
    while slow != fast:
        slow = slow.next
        fast = fast.next
    
    return slow  # This is the cycle start

# Build: 3 → 2 → 0 → -4 → (back to 2)
n1 = ListNode(3)
n2 = ListNode(2)
n3 = ListNode(0)
n4 = ListNode(-4)
n1.next = n2; n2.next = n3; n3.next = n4; n4.next = n2  # Cycle at n2

cycle_start = detectCycle(n1)
print(f"Cycle starts at node with value: {cycle_start.val}")  # 2

---

## 🎯 Problem 4: Palindrome Linked List (LC #234)

Check if a linked list is a palindrome. This beautifully combines BOTH the fast/slow (find middle) and reversal patterns.

```
Input:  1 → 2 → 2 → 1  →  True
Input:  1 → 2         →  False
```

**Strategy:**
1. Find the middle (fast/slow)
2. Reverse the second half
3. Compare first half with reversed second half

In [ ]:
def isPalindrome(head):
    """
    Check palindrome: find middle → reverse second half → compare.
    Time: O(n), Space: O(1)
    """
    if not head or not head.next:
        return True
    
    # Step 1: Find middle (slow stops at start of second half)
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    
    # Step 2: Reverse second half
    prev = None
    curr = slow
    while curr:
        next_temp = curr.next
        curr.next = prev
        prev = curr
        curr = next_temp
    # prev is now head of reversed second half
    
    # Step 3: Compare first half with reversed second half
    left, right = head, prev
    while right:  # right is shorter or equal
        if left.val != right.val:
            return False
        left = left.next
        right = right.next
    
    return True

print(isPalindrome(build_list([1, 2, 2, 1])))     # True
print(isPalindrome(build_list([1, 2, 3, 2, 1])))   # True
print(isPalindrome(build_list([1, 2])))             # False

Look at that — we just combined two fundamental patterns into one solution. This is why learning the building blocks matters.

---

## 🗺️ Two-Pointer Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  LINKED LIST TWO-POINTER PATTERNS                                   ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. FIND MIDDLE: slow=1 step, fast=2 steps                         ║
║     When fast hits end, slow is at middle                            ║
║                                                                      ║
║  2. NTH FROM END: advance fast by n, then move both                 ║
║     When fast hits end, slow is at the target                        ║
║                                                                      ║
║  3. CYCLE START: after detection, reset one to head, walk both ×1   ║
║     They meet at cycle entry (math: distance tail = distance head)  ║
║                                                                      ║
║  4. PALINDROME: find middle + reverse second half + compare         ║
║     Combines two patterns into one solution                          ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Pattern |
|---|---------|------------|--------|
| 876 | Middle of Linked List | Easy | Fast/slow |
| 19 | Remove Nth Node From End | Medium | Gap pointer |
| 142 | Linked List Cycle II | Medium | Floyd's phase 2 |
| 234 | Palindrome Linked List | Easy | Middle + reverse |
| 160 | Intersection of Two Lists | Easy | Two pointer / length diff |
| 202 | Happy Number | Easy | Floyd's on numbers |
| 143 | Reorder List | Medium | Middle + reverse + merge |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Fast/slow finds the middle in one pass — no counting needed    │
├──────────────────────────────────────────────────────────────────────┤
│  📌 "Nth from end" = advance one pointer by n first                │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Cycle start: after meeting, reset to head, walk both at ×1     │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Palindrome = find middle + reverse + compare (3 patterns in 1) │
├──────────────────────────────────────────────────────────────────────┤
│  📌 All these patterns: O(n) time, O(1) space — no extra memory    │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Reversal Patterns

Reversing parts of a list, reversing in groups, and swap-in-pairs.

---

### 🎉 Two-Pointer Mastered!
These patterns power easily 30% of all linked list interview questions.